# Gold — Metric Views dos KPIs (etapa 4.2)

Notebook **comum** (não é fonte de pipeline): cria as três Metric Views que dão aos 8 KPIs oficiais uma definição única, `CREATE OR REPLACE`, então pode ser executado de novo sem efeito colateral. Pré-requisito: o pipeline `olist.gold` rodado com a versão da 4.2 (coluna `pedidos_validos_do_cliente` e a tabela `gold_seller_mensal`).

| Metric View | Fonte | KPIs |
|---|---|---|
| `olist.gold.mv_pedidos` | `fact_orders` + `dim_customer` | GMV, ticket médio, taxa de cancelamento, % entregas no prazo, lead time (média e mediana), review score médio, taxa de recompra; recortes por período, status e UF do cliente |
| `olist.gold.mv_itens` | `fact_order_items` + `dim_seller` + `dim_product` | GMV e ticket médio por categoria, vendedor e UF do vendedor |
| `olist.gold.mv_concentracao` | `gold_seller_mensal` | concentração de vendas (top 10 do mês) |

**Como consultar** uma Metric View: as medidas só existem dentro de `MEASURE(...)`, e as dimensões vão em `GROUP BY`:

```sql
SELECT ano_mes, MEASURE(gmv) AS gmv, MEASURE(pct_entregas_no_prazo) AS pct_no_prazo
FROM olist.gold.mv_pedidos GROUP BY ano_mes ORDER BY ano_mes;
```

Definições completas, com o porquê de cada regra: `docs/dicionario_metricas.md` (v1) e ADR 0007.

## Definições das Metric Views

Cada Metric View é um texto YAML: `source` (a tabela), `joins` (dimensões ligadas pela chave substituta), `dimensions` (recortes) e `measures` (as métricas). Os `FILTER (WHERE ...)` aplicam as flags da `fact_orders` (`pedido_valido`, `entregue`...), e os denominadores usam `NULLIF(..., 0)` para um recorte sem pedidos devolver nulo em vez de erro de divisão por zero.

**Cuidado com o YAML:** um texto com dois-pontos no meio (`comentário: assim`) é lido como outro campo e a criação falha; por isso os comentários evitam `:`. Se precisar de um, ponha o texto entre aspas.

A célula abaixo só guarda os textos em variáveis; nada é criado ainda.

In [0]:
# O QUE FAZ: guarda, em três variáveis de texto, as definições YAML das Metric Views. Nenhum objeto
#   é criado e nenhum dado é lido aqui.
#
# RECURSOS:
# - texto entre ''' ''': string de várias linhas; o YAML depende da indentação, então não a altere.
# - version: 1.1 é a versão do formato YAML das Metric Views.
# - source / joins: a tabela de fatos e as dimensões ligadas pela chave (source.customer_sk = cliente.customer_sk).
# - dimensions: colunas pelas quais se pode agrupar (GROUP BY). measures: agregações oficiais.
# - FILTER (WHERE cond): aplica a agregação só às linhas que cumprem a condição.
# - NULLIF(x, 0): devolve nulo quando x é 0, evitando divisão por zero.
# - percentile(coluna, 0.5): mediana exata. COUNT(DISTINCT x): valores diferentes.
MV_PEDIDOS = '''version: 1.1
comment: KPIs oficiais no grão do pedido (dicionário de métricas v1). Fonte olist.gold.fact_orders.
source: olist.gold.fact_orders
joins:
  - name: cliente
    source: olist.gold.dim_customer
    on: source.customer_sk = cliente.customer_sk
dimensions:
  - name: data_compra
    expr: order_purchase_date
    comment: Data da compra (período de referência de todas as métricas).
  - name: ano_mes
    expr: date_format(order_purchase_date, 'yyyy-MM')
    comment: Mês da compra, no formato aaaa-mm.
  - name: ano
    expr: year(order_purchase_date)
    comment: Ano da compra.
  - name: status_atual
    expr: status_atual
    comment: Status atual do pedido (evento de maior event_sequence).
  - name: uf_cliente
    expr: cliente.customer_state
    comment: UF do cliente na data da compra (versão vigente do cadastro).
measures:
  - name: pedidos_criados
    expr: COUNT(1)
    comment: Todos os pedidos, com ou sem item, em qualquer status.
  - name: pedidos_validos
    expr: COUNT(1) FILTER (WHERE pedido_valido)
    comment: Pedidos com ao menos 1 item e status diferente de canceled e unavailable.
  - name: gmv
    expr: SUM(gmv_bruto) FILTER (WHERE pedido_valido)
    comment: GMV = soma de preço + frete dos itens de pedidos válidos.
  - name: ticket_medio
    expr: SUM(gmv_bruto) FILTER (WHERE pedido_valido) / NULLIF(COUNT(1) FILTER (WHERE pedido_valido), 0)
    comment: GMV dividido pelo número de pedidos válidos.
  - name: pedidos_cancelados
    expr: COUNT(1) FILTER (WHERE cancelado)
    comment: Pedidos com status atual canceled.
  - name: taxa_cancelamento
    expr: COUNT(1) FILTER (WHERE cancelado) / NULLIF(COUNT(1), 0)
    comment: Pedidos canceled dividido por todos os pedidos criados. unavailable não entra.
  - name: pedidos_entregues
    expr: COUNT(1) FILTER (WHERE entregue)
    comment: Status atual delivered e data de entrega preenchida.
  - name: pct_entregas_no_prazo
    expr: COUNT(1) FILTER (WHERE entregue_no_prazo) / NULLIF(COUNT(1) FILTER (WHERE entregue), 0)
    comment: Entregues com DATE(entrega) <= DATE(estimada) dividido pelos entregues.
  - name: lead_time_medio_dias
    expr: AVG(lead_time_dias)
    comment: Média de dias entre a compra e a entrega, só pedidos entregues.
  - name: lead_time_mediana_dias
    expr: percentile(lead_time_dias, 0.5)
    comment: Mediana de dias entre a compra e a entrega, só pedidos entregues.
  - name: review_score_medio
    expr: AVG(review_score) FILTER (WHERE entregue)
    comment: Média da nota da review mais recente dos pedidos entregues.
  - name: clientes_com_compra
    expr: COUNT(DISTINCT customer_unique_id) FILTER (WHERE pedido_valido)
    comment: Clientes únicos com ao menos 1 pedido válido.
  - name: clientes_recorrentes
    expr: COUNT(DISTINCT customer_unique_id) FILTER (WHERE pedido_valido AND pedidos_validos_do_cliente >= 2)
    comment: Clientes únicos com 2 ou mais pedidos válidos (em qualquer período).
  - name: taxa_recompra
    expr: COUNT(DISTINCT customer_unique_id) FILTER (WHERE pedido_valido AND pedidos_validos_do_cliente >= 2) / NULLIF(COUNT(DISTINCT customer_unique_id) FILTER (WHERE pedido_valido), 0)
    comment: Clientes únicos com 2+ pedidos válidos dividido pelos clientes únicos com 1+.
'''

MV_ITENS = '''version: 1.1
comment: GMV e ticket médio no grão do item, para recortar por categoria, vendedor e UF do vendedor. Fonte olist.gold.fact_order_items.
source: olist.gold.fact_order_items
joins:
  - name: vendedor
    source: olist.gold.dim_seller
    on: source.seller_sk = vendedor.seller_sk
  - name: produto
    source: olist.gold.dim_product
    on: source.product_id = produto.product_id
dimensions:
  - name: data_compra
    expr: order_purchase_date
    comment: Data da compra.
  - name: ano_mes
    expr: date_format(order_purchase_date, 'yyyy-MM')
    comment: Mês da compra, no formato aaaa-mm.
  - name: categoria
    expr: produto.categoria
    comment: Categoria do produto em inglês (desconhecida quando não há).
  - name: seller_id
    expr: seller_id
    comment: Vendedor do item.
  - name: uf_vendedor
    expr: vendedor.seller_state
    comment: UF do vendedor na data da compra (versão vigente do cadastro).
measures:
  - name: gmv
    expr: SUM(item_total) FILTER (WHERE pedido_valido)
    comment: GMV = soma de preço + frete dos itens de pedidos válidos. Mesmo número da mv_pedidos.
  - name: itens_vendidos
    expr: COUNT(1) FILTER (WHERE pedido_valido)
    comment: Itens de pedidos válidos.
  - name: pedidos_validos
    expr: COUNT(DISTINCT order_id) FILTER (WHERE pedido_valido)
    comment: Pedidos válidos que têm ao menos um item no recorte (um pedido com 2 vendedores conta nos dois).
  - name: ticket_medio
    expr: SUM(item_total) FILTER (WHERE pedido_valido) / NULLIF(COUNT(DISTINCT order_id) FILTER (WHERE pedido_valido), 0)
    comment: GMV do recorte dividido pelos pedidos válidos do recorte.
'''

MV_CONCENTRACAO = '''version: 1.1
comment: Concentração de vendas, que é o GMV dos 10 maiores vendedores de cada mês dividido pelo GMV do mês. Fonte olist.gold.gold_seller_mensal.
source: olist.gold.gold_seller_mensal
dimensions:
  - name: ano_mes
    expr: ano_mes
    comment: Mês da compra, no formato aaaa-mm.
measures:
  - name: gmv_total
    expr: SUM(gmv)
    comment: GMV de todos os vendedores.
  - name: gmv_top10
    expr: SUM(gmv) FILTER (WHERE rank_no_mes <= 10)
    comment: GMV dos 10 maiores vendedores de cada mês.
  - name: concentracao_top10
    expr: SUM(gmv) FILTER (WHERE rank_no_mes <= 10) / NULLIF(SUM(gmv), 0)
    comment: GMV do top 10 do mês dividido pelo GMV total. Sem recorte, soma mês a mês.
'''

## Criação das Metric Views

`CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML AS $$ ... $$` cria (ou recria) a Metric View. Não grava dado: ela é uma definição que lê a gold no momento da consulta, então sempre mostra o número atual.

In [0]:
# O QUE FAZ: cria as três Metric Views no schema gold a partir dos textos da célula anterior e lista o
#   que foi criado. É idempotente: rodar de novo recria as mesmas definições.
#
# FUNÇÕES DECLARADAS:
# - criar_metric_view(nome, yaml): monta e executa o CREATE OR REPLACE VIEW ... WITH METRICS.
#
# FUNÇÕES NATIVAS:
# - spark.sql(texto): executa um comando SQL e devolve um DataFrame.
# - f"...{variavel}...": f-string, insere a variável no texto. $$ ... $$ delimita o YAML dentro do SQL.
# - for nome, yaml in (...): percorre os pares (nome, texto).
# - spark.sql("SHOW VIEWS IN ..."): lista as views do schema; display(df) mostra a tabela no notebook.
CATALOG = "olist"


def criar_metric_view(nome: str, definicao: str) -> None:
    spark.sql(
        f"CREATE OR REPLACE VIEW {CATALOG}.gold.{nome} WITH METRICS LANGUAGE YAML AS $$\n{definicao}$$"
    )


for nome, definicao in (("mv_pedidos", MV_PEDIDOS), ("mv_itens", MV_ITENS), ("mv_concentracao", MV_CONCENTRACAO)):
    criar_metric_view(nome, definicao)

spark.sql(f"USE CATALOG {CATALOG}")  # SHOW VIEWS não aceita referência entre catálogos; usa o catálogo corrente
display(spark.sql(f"SHOW VIEWS IN gold LIKE 'mv_*'"))

## Conferência rápida

Os 8 KPIs sem recorte, direto das Metric Views. **A prova de que estão certos não é esta célula**: ela só mostra os números. A comparação com um cálculo independente (direto da silver) é o notebook `quality/04_validacao_gold`.

O GMV precisa ser **o mesmo** em `mv_pedidos` e `mv_itens`.

In [0]:
# O QUE FAZ: consulta cada Metric View sem recorte e mostra os KPIs oficiais lado a lado.
#
# FUNÇÕES NATIVAS:
# - MEASURE(nome): avalia a medida da Metric View no recorte do GROUP BY (aqui, sem recorte = total).
# - spark.sql(...).display() / display(df): mostra o resultado como tabela.
display(spark.sql("""
SELECT MEASURE(gmv) AS gmv,
       MEASURE(pedidos_validos) AS pedidos_validos,
       MEASURE(ticket_medio) AS ticket_medio,
       MEASURE(taxa_cancelamento) AS taxa_cancelamento,
       MEASURE(pct_entregas_no_prazo) AS pct_entregas_no_prazo,
       MEASURE(lead_time_medio_dias) AS lead_time_medio_dias,
       MEASURE(lead_time_mediana_dias) AS lead_time_mediana_dias,
       MEASURE(review_score_medio) AS review_score_medio,
       MEASURE(taxa_recompra) AS taxa_recompra
FROM olist.gold.mv_pedidos
"""))

display(spark.sql("SELECT MEASURE(gmv) AS gmv_mv_itens, MEASURE(ticket_medio) AS ticket_mv_itens FROM olist.gold.mv_itens"))
display(spark.sql("SELECT MEASURE(concentracao_top10) AS concentracao_top10 FROM olist.gold.mv_concentracao"))

## Como executar

1. Rode de novo o pipeline `olist.gold` com o `01_gold_modelo` atualizado (coluna `pedidos_validos_do_cliente` em `fact_orders` e a tabela `gold_seller_mensal`). *Run pipeline* normal: `fact_orders` e `fact_order_items` são recalculadas, as dimensões ficam `NO_OP`.
2. Importe este notebook em `src/gold/02_metric_views` e rode todas as células.
3. Me mande o resultado da última célula e o erro, se houver.

**Valores de referência já vistos** (consultas de teste): GMV R$ 15.614.632,11 · % no prazo 93,27% · lead time mediano 10,31 dias · review score médio 4,15 · GMV por categoria: `health_beauty` R$ 1.417.291,22. Taxa de recompra, ticket, cancelamento e concentração saem pela primeira vez aqui.